# Exercício 19 — resumo de chat por utilizador (PostgreSQL)

**Objectivos**

- Persistir um **resumo acumulado** por utilizador (`external_id`) na base de dados.
- Ao **encerrar** uma sessão de chat, gerar **resumo da sessão** e **fundir** com o resumo anterior (via LLM).
- Em cada **nova sessão** (novo `thread_id` / `session_uuid`), injectar o resumo acumulado como **`SystemMessage`** no primeiro turno.

**Requisitos:** `GOOGLE_API_KEY` ou `GEMINI_API_KEY` no `.env` na raiz do repositório; Postgres a correr (`./run_jupyter.sh`). Opcional: `GEMINI_MODEL_EX19`.

**Ficheiros:** `memoria_resumo_db.py` (SQL + persistência), `agente_chat_memoria.py` (LangGraph + Gemini).


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

from dotenv import load_dotenv

for p in (Path.cwd(), *Path.cwd().parents):
    env = p / ".env"
    if env.is_file():
        load_dotenv(env)
        break

here = Path.cwd().resolve()
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

from langchain_core.messages import HumanMessage

from agente_chat_memoria import (
    criar_grafo_chat,
    criar_llm_resumo,
    mensagens_iniciais_sessao,
    mensagens_para_texto,
    config_thread,
)
from memoria_resumo_db import (
    encerrar_sessao_e_persistir_resumos,
    ler_resumo_acumulado,
    listar_sessoes_utilizador,
    obter_ou_criar_utilizador,
    registar_nova_sessao,
)

USER = "ex19-demo-notebook"
graph = criar_grafo_chat()
obter_ou_criar_utilizador(USER)
print("Resumo na BD (início):", (ler_resumo_acumulado(USER) or "(vazio)")[:600])


## 1. Primeira sessão (sem resumo anterior)

Registamos uma linha em `sessoes_chat`, conversamos com o mesmo `thread_id` (memória **só em RAM** via `MemorySaver`), e no fim geramos texto para a BD.

In [ ]:
sess1 = registar_nova_sessao(USER)
cfg1 = config_thread(str(sess1))

r1 = graph.invoke(
    {
        "messages": mensagens_iniciais_sessao(
            USER,
            "Chamo-me João e prefiro respostas muito curtas. O meu projecto chama-se Aurora.",
        )
    },
    cfg1,
)
print(r1["messages"][-1].content)

r2 = graph.invoke(
    {"messages": [HumanMessage(content="Qual é o nome do meu projecto?")]},
    cfg1,
)
print(r2["messages"][-1].content)


## 2. Encerrar sessão → resumo na BD

`encerrar_sessao_e_persistir_resumos` grava `resumo_sessao`, actualiza `resumo_acumulado` (fusão com LLM se já existia texto).

In [ ]:
st = graph.get_state(cfg1)
msgs = st.values["messages"]
tx = mensagens_para_texto(msgs)

llm_sum = criar_llm_resumo(temperature=0.2)
llm_merge = criar_llm_resumo(temperature=0.15)

resumo_sess, acum = encerrar_sessao_e_persistir_resumos(USER, sess1, tx, llm_sum, llm_merge)
print("Resumo desta sessão (primeiros 700 chars):\n", resumo_sess[:700])
print("\n---\nResumo acumulado (primeiros 900 chars):\n", acum[:900])


## 3. Nova sessão — contexto dos chats anteriores

Novo `session_uuid` ⇒ novo `thread_id` no grafo (histórico da sessão anterior **não** está em RAM). O **primeiro** `invoke` usa `mensagens_iniciais_sessao`, que lê `resumo_acumulado` na BD e coloca um `SystemMessage` antes da pergunta.

In [ ]:
sess2 = registar_nova_sessao(USER)
cfg2 = config_thread(str(sess2))

print("Trecho do resumo na BD (injectado no 1º turno):\n", ler_resumo_acumulado(USER)[:500], "\n")

r3 = graph.invoke(
    {
        "messages": mensagens_iniciais_sessao(
            USER,
            "Como me chamo, qual é o meu projecto e como devo formatar as tuas respostas?",
        )
    },
    cfg2,
)
print(r3["messages"][-1].content)


## 4. Consultar sessões guardadas

In [ ]:
for row in listar_sessoes_utilizador(USER, limite=8):
    print(row["session_uuid"], "| encerrada:", row["encerrada_em"])
    prev = (row["resumo_sessao"] or "")[:240].replace("\n", " ")
    print(" ", prev, "..." if len(prev) == 240 else "")
    print()
